# Phase 3: Classification
**MIS 637 B — Group 7**  
6 classifiers: Decision Tree, Neural Network, Naïve Bayes, KNN, Logistic Regression, Random Forest

In [1]:
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, roc_auc_score)
from imblearn.over_sampling import SMOTE

sns.set_theme(style='whitegrid')

X = np.load('../data/X_scaled.npy')
y = np.load('../data/y.npy')

## 1. Train/Test Split + SMOTE (handle class imbalance)

In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
print('After SMOTE:', np.bincount(y_train_res))

After SMOTE: [72326 72326]


## 2. Define Models

In [3]:
models = {
    'Decision Tree (C4.5)': DecisionTreeClassifier(criterion='entropy', max_depth=10, random_state=42),
    'Neural Network (BP)':  MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=300, random_state=42),
    'Naïve Bayes':          GaussianNB(),
    'KNN':                  KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'Logistic Regression':  LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
    'Random Forest':        RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
}

## 3. Train + Evaluate All Models

In [4]:
results = []
cv_results = []
trained_models = {}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train_res, y_train_res)
    trained_models[name] = model

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None

    results.append({
        'Model':     name,
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall':    recall_score(y_test, y_pred, zero_division=0),
        'F1':        f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC':   roc_auc_score(y_test, y_prob) if y_prob is not None else float('nan'),
    })

    # 5-fold stratified CV on F1 (more robust than single split)
    cv_f1 = cross_val_score(model, X, y, cv=skf, scoring='f1', n_jobs=-1)
    cv_results.append({'Model': name, 'CV_F1_mean': cv_f1.mean(), 'CV_F1_std': cv_f1.std()})
    print(f'  CV F1: {cv_f1.mean():.4f} ± {cv_f1.std():.4f}')

results_df = pd.DataFrame(results).set_index('Model').round(4)
cv_df = pd.DataFrame(cv_results).set_index('Model').round(4)
print('\n--- Hold-out Test Results ---')
display(results_df)
print('\n--- 5-Fold CV F1 ---')
display(cv_df)


Training Decision Tree (C4.5)...


  CV F1: 0.0511 ± 0.0036
Training Neural Network (BP)...


  CV F1: 0.1468 ± 0.0064
Training Naïve Bayes...


  CV F1: 0.2031 ± 0.0006
Training KNN...


  CV F1: 0.0362 ± 0.0023
Training Logistic Regression...


...

  CV F1: 0.0072 ± 0.0014



--- Hold-out Test Results ---


,Accuracy,Precision,Recall,F1,ROC-AUC
Model,,,,,
Decision Tree (C4.5),0.8736,0.2841,0.0872,0.1334,0.6404
Neural Network (BP),0.8236,0.1501,0.1246,0.1362,0.5512
Naïve Bayes,0.1642,0.1139,0.9573,0.2036,0.5110
KNN,0.6181,0.1341,0.4439,0.2059,0.5537
Logistic Regression,0.6300,0.1608,0.5491,0.2488,0.6283
Random Forest,0.8886,0.5517,0.0070,0.0139,0.6310



--- 5-Fold CV F1 ---


,CV_F1_mean,CV_F1_std
Model,,
Decision Tree (C4.5),0.0511,0.0036
Neural Network (BP),0.1468,0.0064
Naïve Bayes,0.2031,0.0006
KNN,0.0362,0.0023
Logistic Regression,0.0464,0.0096
Random Forest,0.0072,0.0014


In [5]:
results_df.to_csv('../outputs/model_comparison.csv')
cv_df.to_csv('../outputs/model_cv_scores.csv')

# Save all models
with open('../outputs/models/all_models.pkl', 'wb') as f:
    pickle.dump(trained_models, f)

print('Saved.')


Saved.
